# Displaying a Fuaran tree in a notebook

`fuaran-py` implements the Jupyter rich-display protocol, so a tree evaluated as
the last expression of a cell renders **inline** — no import beyond `fuaran_py`,
and nothing added to the package's dependency set (the protocol is a method name,
`_repr_mimebundle_`, not an IPython import).

This notebook is committed **with its outputs**: it is the record that the
protocol was exercised by a real front end, and `tests/test_notebook_display.py`
reads it back and asserts the recorded bundles are genuine. To re-record it after
changing the display path:

```
pip install nbclient            # into a working venv only — not a package dependency
python -m nbclient examples/notebook_display.ipynb --inplace
```


In [1]:
from fuaran_py import decode_node, encode_node
from fuaran_py.renderer import FUARAN_UI_MIME
from fuaran_py.ui import quick

rows = [
    {"region": "North", "revenue": 1284.50},
    {"region": "South", "revenue": 918.00},
    {"region": "East", "revenue": 1102.25},
]

app = quick.dashboard(
    "Regional revenue",
    quick.metric_strip(rows, label="region", value="revenue"),
    quick.grid(rows),
)

# The last expression of the cell — the front end asks it for a mime bundle.
app

Region,Revenue
North,1284.5
South,918
East,1102.25


In [2]:
bundle = app._repr_mimebundle_()

print("representations offered:", sorted(bundle))
print()
print("text/plain:", bundle["text/plain"])
print()
print("canonical wire (first 160 bytes):")
print(bundle[FUARAN_UI_MIME][:160] + " ...")

representations offered: ['application/vnd.fuaran.ui+json', 'text/html', 'text/plain']

text/plain: Fuaran UI tree 'dashboard-regional-revenue-3c8f74' — Box, 6 nodes (repr() for the full structure)

canonical wire (first 160 bytes):
{"accessibility":{"role":"main"},"id":"dashboard-regional-revenue-3c8f74","kind":{"$type":"Box","children":[{"id":"strip-metrics-969db0","kind":{"$type":"Box"," ...


In [3]:
# The same display path serves a DECODED tree — a tree that arrived over the wire
# from any host — so what a notebook shows and what a server serves cannot drift.
decoded = decode_node(bundle[FUARAN_UI_MIME])

print("decodes:", decoded.ok)
print("re-encodes byte-identically:", encode_node(decoded.value) == bundle[FUARAN_UI_MIME])
print("displays identically:", decoded.value._repr_mimebundle_() == bundle)

decodes: True
re-encodes byte-identically: True
displays identically: True


## What is, and is not, interactive

The output above is **static and read-only**, deliberately, and it is worth being
precise about what that means rather than discovering it:

| | |
|---|---|
| **Renders** | the full node vocabulary the server renderer supports — layout, metrics, tables, headings, prose, and the reference styling that makes it look the same as it does served from a web host |
| **Resolves** | `Binding.Static` values, and a data-bound grid's rows, at render time |
| **Shows a placeholder for** | any other binding — there is no host supplying values, so an unresolved slot renders as an em-dash rather than as a lie |
| **Does not do** | anything on click: a `Button` is inert, a `Toggle` does not toggle, and no `Action` reaches your kernel. There is no script in the output and no channel back |
| **Does not fetch** | anything: the stylesheet is inlined and carries no `url(...)`, and every `href` / `src` in the tree has been through the ambient destination policy, which defaults to deny-non-local |

Re-running a cell produces a **new** output rather than patching the one already
on screen. That is not an oversight of this layer — patching a rendered cell in
place needs a live channel and a client runtime, which is a different problem
with different dependencies.

What *is* cheap today is that a re-run of an unchanged cell produces a
byte-identical tree, because `fuaran_py.ui.quick` derives its ids from labels
rather than positions or counters — so `fuaran_py.ops.diff` over two runs is a
short, typed op script, which is the shape an in-place patch would eventually
carry.
